## Rule-based parser

In [ ]:
import re
from dataclasses import dataclass, field, asdict
from typing import List, Optional


LANGUAGES = [
    'Італійська', 'Англійська', 'Арабська', 'Білоруська',
    'Давньогрецька', 'Для слабозорих', 'Кримськотатарська',
    'Латинська', 'Німецька', 'Польська', 'Сербська',
    'Словацька', "Старослов'янська", 'Українська',
    'Французька', "Церковнослов'янська", 'Шрифт Брайля'
]
LANGUAGE_ALIASES = {
    'англійською': 'Англійська',
    'англійська': 'Англійська',
    'українською': 'Українська',
    'українська': 'Українська',
    'французькою': 'Французька',
    'французька': 'Французька',
    'німецькою': 'Німецька',
    'німецька': 'Німецька',
    'польською': 'Польська',
    'польська': 'Польська',
} | {
    l.lower():l for l in LANGUAGES
}

PERIODS = [
    'Література XIX - поч. XX ст. (до 1918 р)',
    'Література XVII - XVIII ст.',
    'Література XX ст.',
    'Середньовічна література. Відродження',
    'Сучасна література'
]
PERIOD_ALIASES = {
    'сучасна література': 'Сучасна література',
    'сучасні автори': 'Сучасна література',

    '20 століття': 'Література XX ст.',
    'xx століття': 'Література XX ст.',

    '19 століття': 'Література XIX - поч. XX ст. (до 1918 р)',
    'xix століття': 'Література XIX - поч. XX ст. (до 1918 р)',

    'середньовічна': 'Середньовічна література. Відродження',
    'відродження': 'Середньовічна література. Відродження',
} | {
    p.lower():p for p in PERIODS
}

TYPES = [
    'Аудіо',
    'Електронна',
    'Паперова'
]
TYPE_ALIASES = {
    'паперова': 'Паперова',
    'друкована': 'Паперова',
    'електронна': 'Електронна',
    'ebook': 'Електронна',
    'аудіокнига': 'Аудіо',
    'аудіо': 'Аудіо',
} | {
    t.lower():t for t in TYPES
}

MOOD_KEYWORDS = {
    'сумна': 'сумний',
    'сумне': 'сумний',
    'весела': 'веселий',
    'веселе': 'веселий',
    'мотиваційна': 'мотиваційний',
    'атмосферна': 'атмосферний',
    'романтична': 'романтичний',
    'легка': 'легкий',
    'похмура': 'похмурий',
}

THEME_KEYWORDS = {
    'війна': 'війна',
    'кохання': 'кохання',
    'любов': 'кохання',
    'історія': 'історія',
    'фентезі': 'фентезі',
    'магія': 'магія',
    'космос': 'космос',
    'психологія': 'психологія',
}

CATEGORY_KEYWORDS = {
    "детектив": "Детективи",
    "детективи": "Детективи",

    "фентезі": "Фентезі",
    "fantasy": "Фентезі",

    "трилер": "Трилери",
    "трилери": "Трилери",

    "роман": "Романи",
    "романи": "Романи",

    "поезія": "Поезія",

    "есеїстика": "Есеїстика",

    "біографія": "Біографії",
    "біографії": "Біографії",

    "історична проза": "Історична проза",

    "наукова фантастика": "Наукова фантастика",
    "фантастика": "Наукова фантастика",

    "дитяча література": "Дитяча література",
}

@dataclass
class ParsedQuery:
    intent: str = "unknown"

    similar_to_books: List[str] = field(default_factory=list)
    similar_to_authors: List[str] = field(default_factory=list)

    authors: List[str] = field(default_factory=list)
    categories: List[str] = field(default_factory=list)
    series: List[str] = field(default_factory=list)
    publishers: List[str] = field(default_factory=list)

    languages: List[str] = field(default_factory=list)
    periods: List[str] = field(default_factory=list)
    types: List[str] = field(default_factory=list)

    year_from: Optional[int] = None
    year_to: Optional[int] = None

    pages_min: Optional[int] = None
    pages_max: Optional[int] = None

    moods: List[str] = field(default_factory=list)
    themes: List[str] = field(default_factory=list)

    free_text: Optional[str] = None

class RuleBasedParser:

    def parse(self, text: str) -> ParsedQuery:
        result = ParsedQuery()

        lower = text.lower()

        result.intent = self._detect_intent(lower)

        result.languages = self._extract_languages(lower)
        result.types = self._extract_types(lower)

        self._extract_years(lower, result)
        self._extract_pages(lower, result)

        result.moods = self._extract_keywords(
            lower,
            MOOD_KEYWORDS
        )

        result.themes = self._extract_keywords(
            lower,
            THEME_KEYWORDS
        )

        # result.free_text = text

        result.periods = self._extract_keywords(lower, PERIOD_ALIASES)

        result.authors = self._extract_authors(text)

        result.similar_to_books = self._extract_similar_books(text)
        result.similar_to_authors = self._extract_similar_authors(text)

        result.categories = self._extract_categories(lower)

        result.series = self._extract_series(text)
        result.publishers = self._extract_publishers(text)

        return result

    def _detect_intent(self, text: str) -> str:
        text = text.lower()

        if any(x in text for x in [
            'привіт',
            'добрий день',
            'доброго дня',
        ]):
            return 'greeting'

        if any(x in text for x in [
            'сподобалась',
            'подобається',
            'улюблена книга',
        ]):
            return 'feedback_like'

        if any(x in text for x in [
            'не сподобалась',
            'не подобається'
        ]):
            return 'feedback_dislike'

        if any(x in text for x in [
            'схоже на книгу',
            'як книга',
            'аналог книги'
        ]):
            return 'similar_book'

        if any(x in text for x in [
            'порадь',
            'порекомендуй',
            'що почитати',
            'шукаю книгу'
        ]):
            return 'recommend_book'

        if any(x in text for x in [
            'інформація про книгу',
            'розкажи про книгу',
            'деталі книги',
            'опиши книгу'
        ]):
            return 'book_query'

        if any(x in text for x in [
            'інформація про автора',
            'хто написав',
            'розкажи про автора',
            'автор книги'
        ]):
            return 'author_query'

        if any(x in text for x in [
            'так',
            'ні',
            'звісно',
            'правильно',
            'неправильно',
            'мається на увазі'
        ]):
            return 'clarification_answer'

        return 'unknown'

    def _extract_languages(self, text: str) -> List[str]:

        result = []

        for alias, language in LANGUAGE_ALIASES.items():
            if alias in text:
                result.append(language)

        return sorted(set(result))

    def _extract_types(self, text: str) -> List[str]:

        result = []

        for alias, book_type in TYPE_ALIASES.items():
            if alias in text:
                result.append(book_type)

        return sorted(set(result))

    def _extract_years(
        self,
        text: str,
        result: ParsedQuery
    ):

        m = re.search(
            r'з\s*(\d{4})\s*по\s*(\d{4})',
            text
        )

        if m:
            result.year_from = int(m.group(1))
            result.year_to = int(m.group(2))

    def _extract_pages(
        self,
        text: str,
        result: ParsedQuery
    ):

        m = re.search(
            r'від\s*(\d+)\s*стор',
            text
        )

        if m:
            result.pages_min = int(m.group(1))

        m = re.search(
            r'до\s*(\d+)\s*стор',
            text
        )

        if m:
            result.pages_max = int(m.group(1))

    def _extract_similar_books(self, text: str) -> List[str]:

        patterns = [
            r'схож\w*\s+на\s+книг\w*\s+[«"]([^»"]+)[»"]',
            r'як\s+[«"]([^»"]+)[»"]',
            r'на\s+кшталт\s+[«"]([^»"]+)[»"]',
        ]

        result = []

        for pattern in patterns:
            result.extend(re.findall(pattern, text, flags=re.I))

        return list(set(result))

    def _extract_similar_authors(self, text: str) -> List[str]:

        patterns = [
            r'як\s+у\s+([А-ЯІЇЄҐ][а-яіїєґ\'-]+\s+[А-ЯІЇЄҐ][а-яіїєґ\'-]+)',
            r'схож\w*\s+на\s+([А-ЯІЇЄҐ][а-яіїєґ\'-]+\s+[А-ЯІЇЄҐ][а-яіїєґ\'-]+)',
        ]

        result = []

        for pattern in patterns:
            result.extend(re.findall(pattern, text))

        return list(set(result))

    def _extract_series(self, text: str) -> List[str]:

        return re.findall(
            r'сері[яї]\s*[«"]([^»"]+)[»"]',
            text,
            flags=re.I
        )

    def _extract_authors(self, text: str) -> List[str]:

        matches = re.findall(
            r'([А-ЯІЇЄҐ][а-яіїєґ\'-]+\s+[А-ЯІЇЄҐ][а-яіїєґ\'-]+)',
            text
        )

        return list(set(matches))

    def _extract_publishers(self, text: str) -> List[str]:

        return re.findall(
            r'видавництв\w*\s*[«"]([^»"]+)[»"]',
            text,
            flags=re.I
        )

    def _extract_categories(self, text: str) -> List[str]:
        found = []

        for keyword, category in CATEGORY_KEYWORDS.items():
            if keyword in text:
                found.append(category)

        return sorted(set(found))

    def _extract_keywords(
        self,
        text: str,
        mapping: dict
    ) -> List[str]:

        found = []

        for keyword, value in mapping.items():
            if keyword in text:
                found.append(value)

        return sorted(set(found))

In [ ]:
with open('nlu_test_output.txt', 'r') as rd:
    tests = [l.split('->') for l in rd.readlines()]
tests

In [ ]:
results = []
parser = RuleBasedParser()
for inp, utt, _ in tests:
    inp, utt = inp.strip(), utt.strip()
    out = asdict(parser.parse(utt))
    results.append([inp, utt, out])

with open("rule_parser_output.txt", "w", encoding="utf-8") as out_f:
    out_f.writelines([f'{inp} -> {utt} -> {out}\n' for inp, utt, out in results])

## Fuzzy parser

In [ ]:
import sqlite3

conn = sqlite3.connect("../../data/gold/sqlite/books_v3.db")
cursor = conn.cursor()

def get_unique_names(table_name):
    cursor.execute(f"SELECT DISTINCT name FROM {table_name}")
    rows = cursor.fetchall()
    return [row[0] for row in rows]

authors = get_unique_names("authors")
categories = get_unique_names("categories")
series = get_unique_names("series")
publishers = get_unique_names("publishers")
books = get_unique_names("books")

PERIODS = get_unique_names("periods")
TYPES = [
    'Аудіо',
    'Електронна',
    'Паперова',
]
LANGUAGES = [
    'Італійська',
    'Англійська',
    'Арабська',
    'Білоруська',
    'Давньогрецька',
    'Для слабозорих',
    'Кримськотатарська',
    'Латинська',
    'Німецька',
    'Польська',
    'Сербська',
    'Словацька',
    "Старослов'янська",
    'Українська',
    'Французька',
    "Церковнослов'янська",
    'Шрифт Брайля',
]
THEMES = [
    "кохання",
    "війна",
    "самотність",
    "дружба",
    "дорослішання",
    "пригоди",
    "містика",
]
MOODS = [
    "веселий",
    "сумний",
    "атмосферний",
    "напружений",
    "мотиваційний",
]

print("Authors:", len(authors), authors[:10])
print("Categories:", len(categories), categories[:10])
print("Series:", len(series), series[:10])
print("Publishers:", len(publishers), publishers[:10])
print("Books:", len(books), books[:10])
print("Languages:", LANGUAGES)
print("Periods:", PERIODS)
print("Types:", TYPES)

conn.close()

In [ ]:
# DEPRECATED - low performance

# import re
# from dataclasses import dataclass, field, asdict
# from typing import List, Optional
# from rapidfuzz import process

# MOODS = MOOD_KEYWORDS.values()

# THEMES = THEME_KEYWORDS.values()

# @dataclass
# class ParsedQuery:
#     intent: str = "unknown"
#     similar_to_books: List[str] = field(default_factory=list)
#     similar_to_authors: List[str] = field(default_factory=list)
#     authors: List[str] = field(default_factory=list)
#     categories: List[str] = field(default_factory=list)
#     series: List[str] = field(default_factory=list)
#     publishers: List[str] = field(default_factory=list)
#     languages: List[str] = field(default_factory=list)
#     periods: List[str] = field(default_factory=list)
#     types: List[str] = field(default_factory=list)
#     year_from: Optional[int] = None
#     year_to: Optional[int] = None
#     pages_min: Optional[int] = None
#     pages_max: Optional[int] = None
#     moods: List[str] = field(default_factory=list)
#     themes: List[str] = field(default_factory=list)
#     free_text: Optional[str] = None

# def fuzzy_match_entities(text: str, choices: List[str], score_cutoff: int = 80) -> List[str]:
#     matches = process.extract(text, choices, score_cutoff=score_cutoff)
#     return [m[0] for m in matches]

# def parse_utterance(utterance: str) -> ParsedQuery:
#     pq = ParsedQuery()

#     if any(word in utterance.lower() for word in ["порадь", "рекомендуй", "порадьте"]):
#         pq.intent = "recommend_book"

#     pq.authors = fuzzy_match_entities(utterance, authors)
#     pq.categories = fuzzy_match_entities(utterance, categories)
#     pq.series = fuzzy_match_entities(utterance, series)
#     pq.publishers = fuzzy_match_entities(utterance, publishers)
#     pq.similar_to_books = fuzzy_match_entities(utterance, books)
#     pq.periods = fuzzy_match_entities(utterance, PERIODS)
#     pq.types = fuzzy_match_entities(utterance, TYPES)
#     pq.languages = fuzzy_match_entities(utterance, LANGUAGES)
#     pq.moods = fuzzy_match_entities(utterance, MOODS)
#     pq.themes = fuzzy_match_entities(utterance, THEMES)

#     years = [int(y) for y in re.findall(r"\b(1[0-9]{3}|20[0-9]{2}|2100)\b", utterance)]
#     if years:
#         pq.year_from = min(years)
#         pq.year_to = max(years)

#     pages = [int(p) for p in re.findall(r"\b\d{2,4}\b(?=\s*сторін)", utterance)]
#     if pages:
#         pq.pages_min = min(pages)
#         pq.pages_max = max(pages)

#     return pq

# utterance = "порадь книги серії український детектив від ранку чи атени, що вийшли з 1997 по 2017, сторінок до 300"
# parsed = parse_utterance(utterance)
# print(asdict(parsed))

In [ ]:
from dataclasses import dataclass, field, asdict
from typing import List, Optional
import re

from rapidfuzz import process, fuzz
import re
from rapidfuzz import process, fuzz


def normalize(text: str) -> str:
    text = text.lower()

    text = text.replace("’", "'")
    text = text.replace("`", "'")

    text = re.sub(r"[«»\"()]", " ", text)
    text = re.sub(r"[-_/]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def generate_ngrams(
    text: str,
    min_words: int = 1,
    max_words: int = 6
):

    words = normalize(text).split()

    ngrams = []

    for n in range(min_words, max_words + 1):
        for i in range(len(words) - n + 1):
            ngrams.append(
                " ".join(words[i:i+n])
            )

    return ngrams


def exact_dictionary_match(
    text: str,
    dictionary: list[str]
) -> list[str]:

    normalized_text = normalize(text)

    found = []

    for value in dictionary:

        if normalize(value) in normalized_text:
            found.append(value)

    return found


def fuzzy_dictionary_match(
    text: str,
    dictionary: list[str],
    score_cutoff: int = 90
) -> list[str]:
    
    dictionary = {
        normalize(d):d for d in dictionary
    }

    candidates = []

    ngrams = generate_ngrams(text)
    # print('ngrams', ngrams)

    for chunk in ngrams:

        best = process.extractOne(
            chunk,
            dictionary.keys(),
            scorer=fuzz.token_set_ratio,
            score_cutoff=score_cutoff
        )

        if best:
            candidates.append(best)
        # print(chunk, best)

    scores = {}

    for value, score, _ in candidates:

        if value not in scores:
            scores[value] = score
        else:
            scores[value] = max(
                scores[value],
                score
            )

    return [
        dictionary[value]
        for value, score in sorted(
            scores.items(),
            key=lambda x: x[1],
            reverse=True
        )
    ]


def extract_dictionary_entities(
    text: str,
    dictionary: list[str],
    fuzzy_threshold: int = 90
) -> list[str]:

    # 1. exact match

    exact_matches = exact_dictionary_match(
        text,
        dictionary
    )

    if exact_matches:
        return exact_matches

    # 2. fuzzy fallback

    return fuzzy_dictionary_match(
        text,
        dictionary,
        score_cutoff=fuzzy_threshold
    )


def extract_fuzzy_new(
    utterance: str,
    data: list[str],
    fuzzy_threshold=88,
) -> list[str]:

    return extract_dictionary_entities(
        utterance,
        data,
        fuzzy_threshold=fuzzy_threshold,
    )

# CONFIG

AUTHOR_THRESHOLD = 95
BOOK_THRESHOLD = 95
CATEGORY_THRESHOLD = 95
SERIES_THRESHOLD = 95
PUBLISHER_THRESHOLD = 80
OTHER_THRESHOLD = 90

BOOK_TRIGGERS = [
    "схоже на",
    "подібне до",
    "як книга",
    "на кшталт",
]

AUTHOR_TRIGGERS = [
    "як у",
    "як у автора",
    "схоже на автора",
    "подібне до автора",
    "в стилі",
]


@dataclass
class ParsedQuery:
    intent: str = "unknown"

    similar_to_books: List[str] = field(default_factory=list)
    similar_to_authors: List[str] = field(default_factory=list)

    authors: List[str] = field(default_factory=list)
    categories: List[str] = field(default_factory=list)
    series: List[str] = field(default_factory=list)
    publishers: List[str] = field(default_factory=list)

    languages: List[str] = field(default_factory=list)
    periods: List[str] = field(default_factory=list)
    types: List[str] = field(default_factory=list)

    year_from: Optional[int] = None
    year_to: Optional[int] = None

    pages_min: Optional[int] = None
    pages_max: Optional[int] = None

    moods: List[str] = field(default_factory=list)
    themes: List[str] = field(default_factory=list)

    # free_text: str = ""


def unique(items):
    return list(dict.fromkeys(items))


def fuzzy_match_entities(
    text: str,
    choices: List[str],
    score_cutoff: int,
    limit: int = 10,
) -> List[str]:

    if not text.strip():
        return []

    matches = process.extract(
        text,
        choices,
        scorer=fuzz.token_set_ratio,
        score_cutoff=score_cutoff,
        limit=limit,
    )

    return [m[0] for m in matches]


def extract_after_keywords(
    text: str,
    keywords: List[str],
) -> List[str]:

    results = []

    for keyword in keywords:
        pattern = rf"{keyword}\s+([^,.!?;]+)"
        matches = re.findall(pattern, text, flags=re.IGNORECASE)

        for match in matches:
            results.append(match.strip())

    return results


def _detect_intent(text: str) -> str:
        text = text.lower()

        if any(x in text for x in [
            'привіт',
            'добрий день',
            'доброго дня',
        ]):
            return 'greeting'

        if any(x in text for x in [
            'сподобалась',
            'подобається',
            'улюблена книга',
        ]):
            return 'feedback_like'

        if any(x in text for x in [
            'не сподобалась',
            'не подобається'
        ]):
            return 'feedback_dislike'

        if any(x in text for x in [
            'схоже на книгу',
            'як книга',
            'аналог книги'
        ]):
            return 'similar_book'

        if any(x in text for x in [
            'порадь',
            'порекомендуй',
            'що почитати',
            'шукаю книгу',
            "порадьте",
            "рекомендуй",
        ]):
            return 'recommend_book'

        if any(x in text for x in [
            'інформація про книгу',
            'розкажи про книгу',
            'деталі книги',
            'опиши книгу'
        ]):
            return 'book_query'

        if any(x in text for x in [
            'інформація про автора',
            'хто написав',
            'розкажи про автора',
            'автор книги'
        ]):
            return 'author_query'

        if any(x in text for x in [
            'так',
            'ні',
            'звісно',
            'правильно',
            'неправильно',
            'мається на увазі'
        ]):
            return 'clarification_answer'

        return 'unknown'


def parse_utterance(
    utterance: str,
    books: List[str],
    authors: List[str],
    categories: List[str],
    series: List[str],
    publishers: List[str],
    LANGUAGES: List[str],
    PERIODS: List[str],
    TYPES: List[str],
    MOODS: List[str],
    THEMES: List[str],
) -> ParsedQuery:

    pq = ParsedQuery()

    text = utterance.strip()
    lower = text.lower()


    pq.intent = _detect_intent(lower)

    pq.authors = fuzzy_match_entities(
        text,
        authors,
        AUTHOR_THRESHOLD,
    )

    pq.categories = fuzzy_match_entities(
        text,
        categories,
        CATEGORY_THRESHOLD,
    )
    pq.series = extract_fuzzy_new(text, series, SERIES_THRESHOLD)

    pq.publishers = extract_fuzzy_new(text, publishers, PUBLISHER_THRESHOLD)

    if any(trigger in lower for trigger in BOOK_TRIGGERS):

        pq.similar_to_books = fuzzy_match_entities(
            text,
            books,
            BOOK_THRESHOLD,
        )

    if any(trigger in lower for trigger in AUTHOR_TRIGGERS):

        pq.similar_to_authors = fuzzy_match_entities(
            text,
            authors,
            AUTHOR_THRESHOLD,
        )

    pq.periods = fuzzy_match_entities(
        text,
        PERIODS,
        OTHER_THRESHOLD,
    )

    pq.types = fuzzy_match_entities(
        text,
        TYPES,
        OTHER_THRESHOLD,
    )

    pq.languages = fuzzy_match_entities(
        text,
        LANGUAGES,
        OTHER_THRESHOLD,
    )

    pq.moods = fuzzy_match_entities(
        text,
        MOODS,
        OTHER_THRESHOLD,
    )

    pq.themes = fuzzy_match_entities(
        text,
        THEMES,
        OTHER_THRESHOLD,
    )

    year_range = re.search(
        r"з\s*(\d{4})\s*по\s*(\d{4})",
        lower
    )

    if year_range:
        pq.year_from = int(year_range.group(1))
        pq.year_to = int(year_range.group(2))
    else:
        years = [
            int(y)
            for y in re.findall(
                r"\b(1[0-9]{3}|20[0-9]{2}|2100)\b",
                text
            )
        ]

        if years:
            pq.year_from = min(years)
            pq.year_to = max(years)

    m = re.search(
        r"(?:до|не більше)\s*(\d+)\s*(?:сторінок|сторінки|стор)",
        lower
    )

    if m:
        pq.pages_max = int(m.group(1))

    m = re.search(
        r"(?:сторінок|сторінки|стор)\s*до\s*(\d+)",
        lower
    )

    if m:
        pq.pages_max = int(m.group(1))

    m = re.search(
        r"(?:від|не менше)\s*(\d+)\s*(?:сторінок|сторінки|стор)",
        lower
    )

    if m:
        pq.pages_min = int(m.group(1))

    # cleanup

    pq.authors = unique(pq.authors)
    pq.categories = unique(pq.categories)
    pq.series = unique(pq.series)
    pq.publishers = unique(pq.publishers)

    pq.similar_to_books = unique(pq.similar_to_books)
    pq.similar_to_authors = unique(pq.similar_to_authors)

    pq.languages = unique(pq.languages)
    pq.periods = unique(pq.periods)
    pq.types = unique(pq.types)

    pq.moods = unique(pq.moods)
    pq.themes = unique(pq.themes)

    return pq

parsed = parse_utterance(
    "порадь книги серії український детектив від ранку чи атени, що вийшли з 1997 по 2017, сторінок до 300",
    books,
    authors,
    categories,
    series,
    publishers,
    LANGUAGES,
    PERIODS,
    TYPES,
    MOODS,
    THEMES,
)

print(asdict(parsed))

In [ ]:
results = []
i = 0
cnt = len(tests)
for inp, utt, _ in tests:
    i += 1
    print(i, '/', cnt, ')', end=' ')
    inp, utt = inp.strip(), utt.strip()
    out = asdict(parse_utterance(utt,
                                books,
                                authors,
                                categories,
                                series,
                                publishers,
                                LANGUAGES,
                                PERIODS,
                                TYPES,
                                MOODS,
                                THEMES,
    ))
    results.append([inp, utt, out])
    print(utt, out)

with open("fuzzy_parser_output.txt", "w", encoding="utf-8") as out_f:
    out_f.writelines([f'{inp} -> {utt} -> {out}\n' for inp, utt, out in results])

## Combined parser

In [ ]:
import sqlite3

conn = sqlite3.connect("../../data/gold/sqlite/books_v3.db")
cursor = conn.cursor()

def get_unique_names(table_name):
    cursor.execute(f"SELECT DISTINCT name FROM {table_name}")
    rows = cursor.fetchall()
    return [row[0] for row in rows]

authors = get_unique_names("authors")
categories = get_unique_names("categories")
series = get_unique_names("series")
publishers = get_unique_names("publishers")
books = get_unique_names("books")

PERIODS = get_unique_names("periods")
TYPES = [
    'Аудіо',
    'Електронна',
    'Паперова',
]
LANGUAGES = [
    'Італійська',
    'Англійська',
    'Арабська',
    'Білоруська',
    'Давньогрецька',
    'Для слабозорих',
    'Кримськотатарська',
    'Латинська',
    'Німецька',
    'Польська',
    'Сербська',
    'Словацька',
    "Старослов'янська",
    'Українська',
    'Французька',
    "Церковнослов'янська",
    'Шрифт Брайля',
]
THEMES = [
    "кохання",
    "війна",
    "самотність",
    "дружба",
    "дорослішання",
    "пригоди",
    "містика",
]
MOODS = [
    "веселий",
    "сумний",
    "атмосферний",
    "напружений",
    "мотиваційний",
]

print("Authors:", len(authors), authors[:10])
print("Categories:", len(categories), categories[:10])
print("Series:", len(series), series[:10])
print("Publishers:", len(publishers), publishers[:10])
print("Books:", len(books), books[:10])
print("Languages:", LANGUAGES)
print("Periods:", PERIODS)
print("Types:", TYPES)

conn.close()

In [ ]:
import re
from dataclasses import asdict, dataclass, field
from difflib import SequenceMatcher
from typing import Dict, Iterable, List, Optional


LANGUAGE_ALIASES = {
    "англійською": "Англійська",
    "англійська": "Англійська",
    "англомовна": "Англійська",
    "українською": "Українська",
    "українська": "Українська",
    "україномовна": "Українська",
    "французькою": "Французька",
    "французька": "Французька",
    "німецькою": "Німецька",
    "німецька": "Німецька",
    "польською": "Польська",
    "польська": "Польська",
    "італійською": "Італійська",
    "італійська": "Італійська",
    "арабською": "Арабська",
    "арабська": "Арабська",
    **{language.lower(): language for language in LANGUAGES},
}

TYPE_ALIASES = {
    "паперова": "Паперова",
    "паперову": "Паперова",
    "друкована": "Паперова",
    "друковану": "Паперова",
    "фізична": "Паперова",
    "електронна": "Електронна",
    "електронну": "Електронна",
    "ebook": "Електронна",
    "e-book": "Електронна",
    "аудіокнига": "Аудіо",
    "аудіокнигу": "Аудіо",
    "аудіо": "Аудіо",
    **{book_type.lower(): book_type for book_type in TYPES},
}

PERIOD_ALIASES = {
    "сучасна література": "Сучасна література",
    "сучасну літературу": "Сучасна література",
    "сучасні автори": "Сучасна література",
    "новинки": "Сучасна література",
    "20 століття": "Література XX ст.",
    "xx століття": "Література XX ст.",
    "двадцятого століття": "Література XX ст.",
    "19 століття": "Література XIX - поч. XX ст. (до 1918 р)",
    "xix століття": "Література XIX - поч. XX ст. (до 1918 р)",
    "дев'ятнадцятого століття": "Література XIX - поч. XX ст. (до 1918 р)",
    "17 століття": "Література XVII - XVIII ст.",
    "18 століття": "Література XVII - XVIII ст.",
    "середньовічна": "Середньовічна література. Відродження",
    "середньовіччя": "Середньовічна література. Відродження",
    "відродження": "Середньовічна література. Відродження",
    **{period.lower(): period for period in PERIODS},
}

MOOD_ALIASES = {
    "сумна": "сумний",
    "сумне": "сумний",
    "весела": "веселий",
    "веселе": "веселий",
    "мотиваційна": "мотиваційний",
    "мотиваційне": "мотиваційний",
    "атмосферна": "атмосферний",
    "атмосферне": "атмосферний",
    "романтична": "романтичний",
    "романтичне": "романтичний",
    "легка": "легкий",
    "легке": "легкий",
    "похмура": "похмурий",
    "похмуре": "похмурий",
    "напружена": "напружений",
    "напружене": "напружений",
    "іронічна": "іронічний",
    "іронічне": "іронічний",
    **{mood: mood for mood in MOODS},
}

THEME_ALIASES = {
    "війна": "війна",
    "війну": "війна",
    "кохання": "кохання",
    "любов": "кохання",
    "дружба": "дружба",
    "дружбу": "дружба",
    "дорослішання": "дорослішання",
    "пригоди": "пригоди",
    "пригодницька": "пригоди",
    "містика": "містика",
    "містику": "містика",
    "історія": "історія",
    "історію": "історія",
    "фентезі": "фентезі",
    "магія": "магія",
    "магію": "магія",
    "космос": "космос",
    "психологія": "психологія",
    "психологію": "психологія",
}

CATEGORY_ALIASES = {
    c.lower(): c for c in categories
}

BOOK_TRIGGERS = [
    "схоже на",
    "подібне до",
    "як книга",
    "як книжка",
    "аналог книги",
    "на кшталт",
    "у стилі",
    "в стилі",
]

AUTHOR_TRIGGERS = [
    "як у",
    "як у автора",
    "схоже на автора",
    "подібне до автора",
    "у стилі автора",
    "в стилі автора",
]

STOPWORDS = {
    "а",
    "або",
    "без",
    "би",
    "б",
    "в",
    "від",
    "для",
    "до",
    "з",
    "за",
    "і",
    "й",
    "на",
    "не",
    "по",
    "про",
    "та",
    "у",
    "чи",
    "що",
    "як",
    "яку",
    "які",
    "книга",
    "книги",
    "книгу",
    "книжка",
    "книжки",
    "порадь",
    "порадьте",
    "порекомендуй",
    "порекомендуйте",
    "шукаю",
}


@dataclass
class ParsedQuery:
    intent: str = "unknown"
    similar_to_books: List[str] = field(default_factory=list)
    similar_to_authors: List[str] = field(default_factory=list)
    authors: List[str] = field(default_factory=list)
    categories: List[str] = field(default_factory=list)
    series: List[str] = field(default_factory=list)
    publishers: List[str] = field(default_factory=list)
    languages: List[str] = field(default_factory=list)
    periods: List[str] = field(default_factory=list)
    types: List[str] = field(default_factory=list)
    year_from: Optional[int] = None
    year_to: Optional[int] = None
    pages_min: Optional[int] = None
    pages_max: Optional[int] = None
    moods: List[str] = field(default_factory=list)
    themes: List[str] = field(default_factory=list)
    free_text: Optional[str] = None

@dataclass(frozen=True)
class Candidate:
    field: str
    text: str
    source: str
    score: float = 100.0

def normalize(text: str) -> str:
    text = text.lower()
    text = text.replace("’", "'").replace("`", "'").replace("ʼ", "'")
    text = re.sub(r"[«»\"“”(){}\[\],.!?;:]", " ", text)
    text = re.sub(r"[-_/]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def unique(items: Iterable[str]) -> List[str]:
    return list(dict.fromkeys(item for item in items if item))

def token_set_ratio(left: str, right: str) -> float:
    left_tokens = set(normalize(left).split())
    right_tokens = set(normalize(right).split())

    if not left_tokens or not right_tokens:
        return 0.0

    normalized_left = normalize(left)
    normalized_right = normalize(right)
    if normalized_left == normalized_right:
        return 100.0

    left_sorted = " ".join(sorted(left_tokens))
    right_sorted = " ".join(sorted(right_tokens))
    string_score = max(
        SequenceMatcher(None, normalized_left, normalized_right).ratio(),
        SequenceMatcher(None, left_sorted, right_sorted).ratio(),
    ) * 100
    overlap_score = (len(left_tokens & right_tokens) / len(left_tokens | right_tokens)) * 100
    return (string_score * 0.75) + (overlap_score * 0.25)

def generate_ngrams(text: str, min_words: int = 1, max_words: int = 6) -> List[str]:
    words = [word for word in normalize(text).split() if word not in STOPWORDS]
    ngrams = []
    for size in range(min_words, max_words + 1):
        for index in range(len(words) - size + 1):
            ngrams.append(" ".join(words[index:index + size]))
    return ngrams


class CombinedParser:
    def __init__(
        self,
        entities: Optional[Dict[str, List[str]]] = None,
    ):
        self.entities = {}

        if entities:
            for key, values in entities.items():
                self.entities[key] = unique(values)

        self.normalized_entities = {
            key: {normalize(value): value for value in values}
            for key, values in self.entities.items()
        }
        self.token_index = {
            key: self._build_token_index(values)
            for key, values in self.entities.items()
        }

    def parse(self, utterance: str, include_candidates: bool = False):
        candidates = self.extract_candidates(utterance)
        result = ParsedQuery(intent=self._detect_intent(utterance), free_text=utterance)

        result.languages = self._resolve_alias_field(candidates, "languages")
        result.types = self._resolve_alias_field(candidates, "types")
        result.periods = self._resolve_field(candidates, "periods", "periods", 70)
        result.moods = self._resolve_field(candidates, "moods", "moods", 70)
        result.themes = self._resolve_field(candidates, "themes", "themes", 70)
        result.categories = self._resolve_field(candidates, "categories", "categories", 70)
        result.authors = self._resolve_field(candidates, "authors", "authors", 70)
        result.series = self._resolve_field(candidates, "series", "series", 70)
        result.publishers = self._resolve_field(candidates, "publishers", "publishers", 70)
        result.similar_to_books = self._resolve_field(
            candidates,
            "similar_to_books",
            "books",
            70,
        )
        result.similar_to_authors = self._resolve_field(
            candidates,
            "similar_to_authors",
            "authors",
            70,
        )

        self._extract_years(utterance, result)
        self._extract_pages(utterance, result)

        if include_candidates:
            return {
                "parsed": asdict(result),
                "candidates": {
                    field_name: [asdict(candidate) for candidate in field_candidates]
                    for field_name, field_candidates in candidates.items()
                },
            }

        return result

    def extract_candidates(self, utterance: str) -> Dict[str, List[Candidate]]:
        text = utterance.strip()
        lower = normalize(text)
        candidates: Dict[str, List[Candidate]] = {}

        self._add_alias_candidates(candidates, lower, "languages", LANGUAGE_ALIASES)
        self._add_alias_candidates(candidates, lower, "types", TYPE_ALIASES)
        self._add_alias_candidates(candidates, lower, "periods", PERIOD_ALIASES)
        self._add_alias_candidates(candidates, lower, "moods", MOOD_ALIASES)
        self._add_alias_candidates(candidates, lower, "themes", THEME_ALIASES)
        self._add_alias_candidates(candidates, lower, "categories", CATEGORY_ALIASES)

        self._add_regex_candidates(
            candidates,
            text,
            "similar_to_books",
            [
                r"(?:схож\w*\s+на|подібн\w*\s+до|аналог\w*|на\s+кшталт|як\s+книг\w*)\s+(?:книг\w*\s+)?[«\"]([^»\"]+)[»\"]",
                r"(?:схож\w*\s+на|подібн\w*\s+до|аналог\w*|на\s+кшталт|як\s+книг\w*)\s+([^,.!?;]+)",
            ],
            "similar-template",
        )
        self._add_regex_candidates(
            candidates,
            text,
            "similar_to_authors",
            [
                r"(?i:як\s+у|схож\w*\s+на|подібн\w*\s+до|у\s+стилі|в\s+стилі)\s+(?i:автор\w*\s+)?([А-ЯІЇЄҐ][а-яіїєґ'-]+(?:\s+[А-ЯІЇЄҐ][а-яіїєґ'-]+){1,2})",
            ],
            "similar-author-template",
            flags=0,
        )
        self._add_regex_candidates(
            candidates,
            text,
            "authors",
            [
                r"(?i:автор\w*|письменник\w*|письменниц\w*|написан\w*)\s+([А-ЯІЇЄҐ][а-яіїєґ'-]+(?:\s+[А-ЯІЇЄҐ][а-яіїєґ'-]+){1,2})",
                r"([А-ЯІЇЄҐ][а-яіїєґ'-]+\s+[А-ЯІЇЄҐ][а-яіїєґ'-]+)",
            ],
            "author-template",
            flags=0,
        )
        self._add_regex_candidates(
            candidates,
            text,
            "series",
            [
                r"(?:сері[яї]|цикл\w*)\s*[«\"]([^»\"]+)[»\"]",
                r"(?:із|з|у|в)\s+сері[їі]\s+([^,.!?;]+)",
                r"(?:сері[яї]|цикл\w*)\s+([^,.!?;]+)",
            ],
            "series-template",
        )
        self._add_regex_candidates(
            candidates,
            text,
            "publishers",
            [
                r"видавництв\w*\s*[«\"]([^»\"]+)[»\"]",
                r"(?:від|видавництв\w*)\s+([^,.!?;]+)",
            ],
            "publisher-template",
        )

        for field_name, entity_key in [
            ("authors", "authors"),
            ("categories", "categories"),
            ("series", "series"),
            ("publishers", "publishers"),
        ]:
            self._add_exact_db_candidates(candidates, lower, field_name, entity_key)
            self._add_ngram_db_candidates(candidates, text, field_name, entity_key)

        if any(trigger in lower for trigger in BOOK_TRIGGERS):
            self._add_ngram_db_candidates(candidates, text, "similar_to_books", "books")

        if any(trigger in lower for trigger in AUTHOR_TRIGGERS):
            self._add_ngram_db_candidates(candidates, text, "similar_to_authors", "authors")

        return {
            field_name: self._unique_candidates(field_candidates)
            for field_name, field_candidates in candidates.items()
        }

    def _resolve_field(
        self,
        candidates: Dict[str, List[Candidate]],
        field_name: str,
        entity_key: str,
        threshold: float,
    ) -> List[str]:
        field_candidates = candidates.get(field_name, [])
        choices = self.entities.get(entity_key, [])

        if not field_candidates:
            return []

        if not choices:
            return unique(candidate.text for candidate in field_candidates)

        scored = {}
        normalized_choices = self.normalized_entities.get(entity_key, {})

        for candidate in field_candidates:
            normalized_candidate = normalize(candidate.text)
            if normalized_candidate in normalized_choices:
                value = normalized_choices[normalized_candidate]
                scored[value] = max(scored.get(value, 0), 100)
                continue

            best_choice = None
            best_score = 0.0
            for choice in self._choices_for_text(entity_key, candidate.text):
                score = token_set_ratio(candidate.text, choice)
                weighted_score = (score * 0.8) + (candidate.score * 0.2)
                if weighted_score > best_score:
                    best_choice = choice
                    best_score = weighted_score

            if best_choice and best_score >= threshold:
                scored[best_choice] = max(scored.get(best_choice, 0), best_score)

        return [
            value
            for value, _ in sorted(scored.items(), key=lambda item: item[1], reverse=True)
        ][:10]

    def _resolve_alias_field(
        self,
        candidates: Dict[str, List[Candidate]],
        field_name: str,
    ) -> List[str]:
        return unique(candidate.text for candidate in candidates.get(field_name, []))

    def _add_alias_candidates(
        self,
        candidates: Dict[str, List[Candidate]],
        lower: str,
        field_name: str,
        aliases: Dict[str, str],
    ) -> None:
        for alias, value in aliases.items():
            if self._contains_phrase(lower, alias):
                candidates.setdefault(field_name, []).append(
                    Candidate(field_name, value, "alias", 100)
                )

    def _add_regex_candidates(
        self,
        candidates: Dict[str, List[Candidate]],
        text: str,
        field_name: str,
        patterns: List[str],
        source: str,
        flags: int = re.IGNORECASE,
    ) -> None:
        for pattern in patterns:
            for match in re.findall(pattern, text, flags=flags):
                clean = self._clean_candidate(match)
                if clean:
                    candidates.setdefault(field_name, []).append(
                        Candidate(field_name, clean, source, 92)
                    )

    def _add_exact_db_candidates(
        self,
        candidates: Dict[str, List[Candidate]],
        lower: str,
        field_name: str,
        entity_key: str,
    ) -> None:
        for value in self.entities.get(entity_key, []):
            if self._contains_phrase(lower, value):
                candidates.setdefault(field_name, []).append(
                    Candidate(field_name, value, "db-exact", 100)
                )

    def _add_ngram_db_candidates(
        self,
        candidates: Dict[str, List[Candidate]],
        text: str,
        field_name: str,
        entity_key: str,
    ) -> None:
        choices = self.entities.get(entity_key, [])
        if not choices:
            return

        threshold = {
            "authors": 88,
            "categories": 86,
            "series": 82,
            "publishers": 78,
            "similar_to_books": 82,
            "similar_to_authors": 88,
        }.get(field_name, 86)

        best_by_choice = {}
        for chunk in generate_ngrams(text):
            if len(chunk) < 3:
                continue
            plausible_choices = self._choices_for_text(entity_key, chunk)
            if not plausible_choices:
                continue

            best_choice = None
            best_score = 0.0
            for choice in plausible_choices:
                score = token_set_ratio(chunk, choice)
                if score > best_score:
                    best_choice = choice
                    best_score = score

            if best_choice and best_score >= threshold:
                best_by_choice[best_choice] = max(best_by_choice.get(best_choice, 0), best_score)

        for value, score in best_by_choice.items():
            candidates.setdefault(field_name, []).append(
                Candidate(field_name, value, "db-ngram", score)
            )

    def _choices_for_text(self, entity_key: str, text: str, limit: int = 500) -> List[str]:
        tokens = [
            token
            for token in normalize(text).split()
            if len(token) > 2 and token not in STOPWORDS
        ]
        if not tokens:
            return []

        ranked = {}
        index = self.token_index.get(entity_key, {})
        for token in tokens:
            for value in index.get(token, []):
                ranked[value] = ranked.get(value, 0) + 3

            if len(token) >= 5:
                prefix = token[:5]
                for indexed_token, values in index.items():
                    if indexed_token.startswith(prefix) or token.startswith(indexed_token[:5]):
                        for value in values[:50]:
                            ranked[value] = ranked.get(value, 0) + 1

        return [
            value
            for value, _ in sorted(ranked.items(), key=lambda item: item[1], reverse=True)
        ][:limit]

    def _detect_intent(self, text: str) -> str:
        lower = normalize(text)

        if any(item in lower for item in ["привіт", "добрий день", "доброго дня"]):
            return "greeting"
        if any(item in lower for item in ["не сподобалась", "не подобається"]):
            return "feedback_dislike"
        if any(item in lower for item in ["сподобалась", "подобається", "улюблена книга"]):
            return "feedback_like"
        if any(
            item in lower
            for item in [
                "порадь",
                "порадьте",
                "порекомендуй",
                "порекомендуйте",
                "що почитати",
                "шукаю книгу",
                "шукаю книжку",
                "рекомендуй",
            ]
        ):
            return "recommend_book"
        if any(item in lower for item in BOOK_TRIGGERS):
            return "similar_book"
        if any(
            item in lower
            for item in [
                "інформація про книгу",
                "розкажи про книгу",
                "деталі книги",
                "опиши книгу",
                "про що книга",
            ]
        ):
            return "book_query"
        if any(
            item in lower
            for item in [
                "інформація про автора",
                "хто написав",
                "розкажи про автора",
                "автор книги",
            ]
        ):
            return "author_query"
        if any(
            item in lower
            for item in ["так", "ні", "звісно", "правильно", "неправильно", "мається на увазі"]
        ):
            return "clarification_answer"
        return "unknown"

    def _extract_years(self, utterance: str, result: ParsedQuery) -> None:
        lower = normalize(utterance)
        patterns = [
            r"з\s*(\d{4})\s*(?:по|до|-)\s*(\d{4})",
            r"від\s*(\d{4})\s*(?:по|до|-)\s*(\d{4})",
            r"між\s*(\d{4})\s*(?:і|та|-)\s*(\d{4})",
        ]

        for pattern in patterns:
            match = re.search(pattern, lower)
            if match:
                years = sorted([int(match.group(1)), int(match.group(2))])
                result.year_from, result.year_to = years
                return

        after = re.search(r"(?:після|від|з)\s*(\d{4})", lower)
        before = re.search(r"(?:до|перед|не\s+пізніше)\s*(\d{4})", lower)

        if after:
            result.year_from = int(after.group(1))
        if before:
            result.year_to = int(before.group(1))

        if not after and not before:
            years = [int(year) for year in re.findall(r"\b(1[0-9]{3}|20[0-9]{2}|2100)\b", lower)]
            if years:
                result.year_from = min(years)
                result.year_to = max(years)

    def _extract_pages(self, utterance: str, result: ParsedQuery) -> None:
        lower = normalize(utterance)
        page_word = r"(?:сторінок|сторінки|сторінку|стор|с)"

        max_match = re.search(rf"(?:до|не більше|максимум)\s*(\d+)\s*{page_word}", lower)
        if max_match:
            result.pages_max = int(max_match.group(1))

        reverse_max_match = re.search(rf"{page_word}\s*(?:до|не більше|максимум)\s*(\d+)", lower)
        if reverse_max_match:
            result.pages_max = int(reverse_max_match.group(1))

        min_match = re.search(rf"(?:від|не менше|мінімум)\s*(\d+)\s*{page_word}", lower)
        if min_match:
            result.pages_min = int(min_match.group(1))

        reverse_min_match = re.search(rf"{page_word}\s*(?:від|не менше|мінімум)\s*(\d+)", lower)
        if reverse_min_match:
            result.pages_min = int(reverse_min_match.group(1))

        range_match = re.search(rf"(\d+)\s*[-–]\s*(\d+)\s*{page_word}", lower)
        if range_match:
            pages = sorted([int(range_match.group(1)), int(range_match.group(2))])
            result.pages_min, result.pages_max = pages

    @staticmethod
    def _build_token_index(values: List[str]) -> Dict[str, List[str]]:
        index: Dict[str, List[str]] = {}
        for value in values:
            for token in normalize(value).split():
                if len(token) > 2 and token not in STOPWORDS:
                    index.setdefault(token, []).append(value)
        return index

    @staticmethod
    def _contains_phrase(text: str, phrase: str) -> bool:
        normalized_phrase = normalize(phrase)
        if not normalized_phrase:
            return False
        return re.search(rf"(?<!\w){re.escape(normalized_phrase)}(?!\w)", text) is not None

    @staticmethod
    def _clean_candidate(text: str) -> str:
        text = re.split(r"\s+(?:або|чи|та|і)\s+", text.strip(), maxsplit=1, flags=re.IGNORECASE)[0]
        text = re.sub(r"^(?:книга|книгу|книжка|книжку|автора|авторки)\s+", "", text, flags=re.I)
        text = re.sub(r"\s+", " ", text)
        return text.strip(" ,.!?;:«»\"'")

    @staticmethod
    def _unique_candidates(candidates: List[Candidate]) -> List[Candidate]:
        best = {}
        for candidate in candidates:
            key = (candidate.field, normalize(candidate.text))
            if key not in best or candidate.score > best[key].score:
                best[key] = candidate
        return sorted(best.values(), key=lambda item: item.score, reverse=True)

parser = CombinedParser(entities={
            "books": books,
            "authors": authors,
            "categories": categories,
            "series": series,
            "publishers": publishers,
            "periods": PERIODS,
            "languages": LANGUAGES,
            "types": TYPES,
            "moods": MOODS,
            "themes": THEMES,
        })
parsed = parser.parse(
    # "порадь книгу категорії український детектив від видавництва Ранок, "
    # "схоже на книгу «Місто», з 1997 по 2017, сторінок до 300",
    "Що можеш розповісти про Володимира Ткача та Олексія Кононенка?",
    include_candidates=True,
)
print(parsed)

In [ ]:
results = []
i = 0
cnt = len(tests)
for inp, utt, _ in tests:
    i += 1
    print(i, '/', cnt, ')', end=' ')
    inp, utt = inp.strip(), utt.strip()
    # out = parser.parse(utt, include_candidates=True)
    out = asdict(parser.parse(utt))
    results.append([inp, utt, out])
    print(utt, out)

with open("combined_parser_output.txt", "w", encoding="utf-8") as out_f:
    out_f.writelines([f'{inp} -> {utt} -> {out}\n' for inp, utt, out in results])

In [ ]:
parser.entities